# Call Transcript Intent Analysis
### Mobile Device & App Forensics — World-Class Investigative AI Notebook

---

**Notebook file:** `mobile_device_app_forensics/10_call_transcript_intent_analysis.ipynb`

**Why this matters:** Analysts reviewing intercepted or lawfully collected call transcripts need to separate planning chatter, active execution signals, post-event cleanup, and coded exchanges quickly enough to drive preservation, subpoena, and device-examination decisions. This notebook upgrades the original PyTorch draft into a full investigative workflow with synthetic evidence, a transparent baseline, a PyTorch-first neural model with a built-in fallback, explainability, and an analyst-facing triage function.


## Investigative Context

| Dimension | Details |
| --- | --- |
| Topic focus | Call, app, geospatial, and device-state analytics for mobile-centric investigations. |
| Investigative question | How can investigators use AI to classify **police call transcripts** into actionable intent categories without losing forensic context? |
| Operational decision | whether to preserve live intercept windows, subpoena additional carrier metadata, expand device imaging, or prioritize transcript translation |
| Target label | `conversation_intent` |
| Learning goal | Learn how to move from transcript evidence profiling to baseline comparison, neural modeling, explainability, and analyst handoff in a defensible workflow. |

## Evidence Sources

| Source | Why It Matters | Caveat |
| --- | --- | --- |
| Lawfully collected call transcripts | Reveals planning language, urgency shifts, and coded terminology | Transcription noise and slang can obscure meaning |
| Call detail records | Supplies timing, repetition, and contact-frequency context | Carrier rounding can blur exact duration |
| Tower and handset context | Helps distinguish static coordination from movement-linked execution | Tower density and handoffs vary by geography |
| Case notes and prior flags | Adds investigative memory and repeat-actor context | Historic labels can introduce analyst bias |

## Standards and References

- NIST mobile device forensics guidance
- Call detail record analysis practices
- Lawful intercept and transcript handling playbooks
- Educational prototype only: not a substitute for legal review, chain-of-custody procedures, or validated evidentiary tooling


## Step 0 - Imports and Case Framing

Set up the notebook, define the forensic modeling contract, and keep the runtime simple enough for Colab or local Jupyter.


In [ ]:
# Uncomment in Colab if needed:
# !pip install numpy pandas scikit-learn matplotlib seaborn torch

from __future__ import annotations

import copy
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ModuleNotFoundError:
    torch = None
    nn = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RNG = np.random.default_rng(42)
if TORCH_AVAILABLE:
    torch.manual_seed(42)

NOTEBOOK_TITLE = 'Police Call Transcript Intent Analysis'
TOPIC_NAME = 'Mobile Device & App Forensics'
TASK_TYPE = 'multiclass'
TARGET = 'conversation_intent'
CLASS_LABELS = ['planning', 'execution_signal', 'post_event', 'coded_exchange']
FOCUS_LABEL = 'coded_exchange'
DEVICE = 'cuda' if TORCH_AVAILABLE and torch.cuda.is_available() else 'cpu'
ADVANCED_BACKEND = 'PyTorch MLP' if TORCH_AVAILABLE else 'sklearn MLP fallback'

RAW_NUMERIC_FEATURES = [
    'call_hour',
    'call_duration_seconds',
    'repeat_contact_count',
    'tower_handoff_count',
    'prior_case_mentions',
    'silence_ratio',
    'speaker_overlap_ratio',
]
CATEGORICAL_FEATURES = ['line_type', 'language_pattern', 'contact_relationship']
ENGINEERED_FEATURES = [
    'transcript_token_count',
    'urgent_term_count',
    'coded_term_count',
    'coordination_pressure_index',
    'covert_language_index',
    'night_activity_flag',
]
MODEL_FEATURES = ['transcript'] + RAW_NUMERIC_FEATURES + CATEGORICAL_FEATURES + ENGINEERED_FEATURES

print(f'Notebook: {NOTEBOOK_TITLE}')
print(f'Task type: {TASK_TYPE}')
print(f'Target column: {TARGET}')
print(f'Advanced backend: {ADVANCED_BACKEND}')
print(f'Device: {DEVICE}')


## Step 1 - Build a Synthetic Yet Forensically Plausible Dataset

Create call transcripts and context features that reflect investigative shape while staying safe to share.


In [ ]:
LINE_TYPES = ['direct_mobile', 'burner_phone', 'voip_app', 'conference_bridge']
LANGUAGE_PATTERNS = ['single_language', 'mixed_slang', 'code_switched']
CONTACT_RELATIONSHIPS = ['known_contact', 'new_contact', 'hub_contact']

INTENT_PHRASES = {
    'planning': [
        'check the route before midnight',
        'confirm the usual contact point',
        'keep the vehicle near the back entrance',
        'we need the meeting window confirmed',
        'verify who is carrying the spare device',
        'send the timing once the road clears',
    ],
    'execution_signal': [
        'the person is outside now',
        'move once the line goes quiet',
        'we are at the pickup point already',
        'switch to the next handset immediately',
        'the driver is waiting at the corner',
        'start moving as soon as they arrive',
    ],
    'post_event': [
        'clear the chat history tonight',
        'leave the handset off after this',
        'change the route and drop the jacket',
        'wipe the note after reading it',
        'do not call back from the old number',
        'hide the card and stay away from the station',
    ],
    'coded_exchange': [
        'the green folders are ready for pickup',
        'two tickets need moving after rain',
        'bring the blue package to the quiet place',
        'the books are waiting with the night clerk',
        'deliver three umbrellas when the lights dip',
        'the spare keys belong in locker seven',
    ],
}
FILLER_PHRASES = [
    'call me when you are clear',
    'keep this short and routine',
    'do not mention names on this line',
    'the traffic is slow near the bridge',
    'wait for the update before moving',
    'I will send the time separately',
]
URGENCY_VOCAB = {'now', 'immediately', 'tonight', 'short', 'arrive', 'moving', 'move', 'waiting', 'clear'}
CODED_VOCAB = {'green', 'blue', 'package', 'folders', 'tickets', 'books', 'umbrellas', 'keys', 'locker'}

INTENT_CONFIG = {
    'planning': {
        'call_hour_mean': 21.0,
        'duration_mean': 168,
        'duration_sd': 34,
        'repeat_lambda': 5.6,
        'tower_lambda': 1.7,
        'prior_lambda': 1.0,
        'silence_mean': 0.18,
        'overlap_mean': 0.20,
        'line_probs': [0.34, 0.12, 0.18, 0.36],
        'language_probs': [0.50, 0.31, 0.19],
        'contact_probs': [0.52, 0.10, 0.38],
    },
    'execution_signal': {
        'call_hour_mean': 1.4,
        'duration_mean': 92,
        'duration_sd': 24,
        'repeat_lambda': 2.7,
        'tower_lambda': 4.4,
        'prior_lambda': 1.8,
        'silence_mean': 0.10,
        'overlap_mean': 0.31,
        'line_probs': [0.20, 0.26, 0.18, 0.36],
        'language_probs': [0.33, 0.24, 0.43],
        'contact_probs': [0.43, 0.14, 0.43],
    },
    'post_event': {
        'call_hour_mean': 2.2,
        'duration_mean': 70,
        'duration_sd': 18,
        'repeat_lambda': 1.9,
        'tower_lambda': 3.1,
        'prior_lambda': 2.1,
        'silence_mean': 0.24,
        'overlap_mean': 0.13,
        'line_probs': [0.18, 0.38, 0.25, 0.19],
        'language_probs': [0.27, 0.30, 0.43],
        'contact_probs': [0.55, 0.24, 0.21],
    },
    'coded_exchange': {
        'call_hour_mean': 23.2,
        'duration_mean': 132,
        'duration_sd': 27,
        'repeat_lambda': 3.6,
        'tower_lambda': 1.3,
        'prior_lambda': 0.9,
        'silence_mean': 0.29,
        'overlap_mean': 0.16,
        'line_probs': [0.14, 0.34, 0.36, 0.16],
        'language_probs': [0.19, 0.25, 0.56],
        'contact_probs': [0.31, 0.23, 0.46],
    },
}

def clipped_normal(mean, sd, low, high):
    return float(np.clip(RNG.normal(mean, sd), low, high))

def sample_transcript(intent: str) -> str:
    parts = list(RNG.choice(INTENT_PHRASES[intent], size=2, replace=True))
    if RNG.random() < 0.65:
        parts.append(str(RNG.choice(FILLER_PHRASES)))
    RNG.shuffle(parts)
    return ' '.join(parts).lower()

records = []
intent_mix = np.array([0.31, 0.24, 0.22, 0.23])
n_samples = 2200

for case_idx in range(n_samples):
    intent = str(RNG.choice(CLASS_LABELS, p=intent_mix))
    cfg = INTENT_CONFIG[intent]
    call_hour = int(np.round(clipped_normal(cfg['call_hour_mean'], 2.8, 0, 23)))
    call_duration_seconds = int(np.round(clipped_normal(cfg['duration_mean'], cfg['duration_sd'], 20, 420)))
    repeat_contact_count = int(RNG.poisson(cfg['repeat_lambda']))
    tower_handoff_count = int(RNG.poisson(cfg['tower_lambda']))
    prior_case_mentions = int(RNG.poisson(cfg['prior_lambda']))
    silence_ratio = float(np.clip(RNG.normal(cfg['silence_mean'], 0.06), 0.02, 0.75))
    speaker_overlap_ratio = float(np.clip(RNG.normal(cfg['overlap_mean'], 0.05), 0.00, 0.65))
    line_type = str(RNG.choice(LINE_TYPES, p=cfg['line_probs']))
    language_pattern = str(RNG.choice(LANGUAGE_PATTERNS, p=cfg['language_probs']))
    contact_relationship = str(RNG.choice(CONTACT_RELATIONSHIPS, p=cfg['contact_probs']))
    transcript = sample_transcript(intent)

    records.append(
        {
            'case_id': f'PCALL-{case_idx:05d}',
            'call_hour': call_hour,
            'call_duration_seconds': call_duration_seconds,
            'repeat_contact_count': repeat_contact_count,
            'tower_handoff_count': tower_handoff_count,
            'prior_case_mentions': prior_case_mentions,
            'silence_ratio': round(silence_ratio, 3),
            'speaker_overlap_ratio': round(speaker_overlap_ratio, 3),
            'line_type': line_type,
            'language_pattern': language_pattern,
            'contact_relationship': contact_relationship,
            'transcript': transcript,
            TARGET: intent,
        }
    )

df = pd.DataFrame(records)
print(df.head(4).to_string(index=False))
print()
print('Dataset shape:', df.shape)
print(df[TARGET].value_counts().to_string())


## Step 2 - Evidence Profiling

Quantify the transcript mix, metadata balance, and class distribution before modeling.


In [ ]:
print('Numeric evidence summary:')
print(df[RAW_NUMERIC_FEATURES].describe().T[['mean', 'std', 'min', 'max']].round(3).to_string())
print()
print('Categorical evidence mix:')
for feature in CATEGORICAL_FEATURES:
    print(f'\n{feature}:')
    print(df[feature].value_counts(normalize=True).round(3).to_string())

transcript_preview = df[[TARGET, 'transcript']].sample(6, random_state=42).reset_index(drop=True)
print('\nTranscript preview:')
print(transcript_preview.to_string(index=False))


## Step 3 - Exploratory Visual Analytics

Look for timing shifts, transcript-length differences, and evidence separation patterns visually.


In [ ]:
temp_lengths = df['transcript'].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df[TARGET].value_counts(normalize=True).reindex(CLASS_LABELS).plot(kind='bar', ax=axes[0], color=['#4C72B0', '#55A868', '#C44E52', '#8172B3'])
axes[0].set_title('Intent mix')
axes[0].set_ylabel('Share')

sns.boxplot(data=df, x=TARGET, y='call_hour', order=CLASS_LABELS, ax=axes[1], color='#88CCEE')
axes[1].set_title('Call hour by intent')
axes[1].tick_params(axis='x', rotation=20)

sns.histplot(data=df.assign(transcript_token_count=temp_lengths), x='transcript_token_count', hue=TARGET, multiple='stack', bins=18, ax=axes[2])
axes[2].set_title('Transcript length by intent')
plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.scatterplot(
    data=df.sample(500, random_state=42),
    x='repeat_contact_count',
    y='tower_handoff_count',
    hue=TARGET,
    alpha=0.72,
    ax=axes[0],
)
axes[0].set_title('Repeat contact vs tower handoff')

corr = df[RAW_NUMERIC_FEATURES].corr().round(2)
sns.heatmap(corr, cmap='vlag', center=0, annot=True, fmt='.2f', ax=axes[1])
axes[1].set_title('Metadata correlation')
plt.tight_layout()


## Step 4 - Feature Engineering

Convert transcript evidence and call metadata into explicit forensic signals an analyst can reason about.


In [ ]:
def tokenize(text: str):
    return re.findall(r"[a-z']+", text.lower())

def count_hits(text: str, vocabulary: set[str]) -> int:
    return sum(token in vocabulary for token in tokenize(text))

df['transcript_token_count'] = df['transcript'].apply(lambda text: len(tokenize(text)))
df['urgent_term_count'] = df['transcript'].apply(lambda text: count_hits(text, URGENCY_VOCAB))
df['coded_term_count'] = df['transcript'].apply(lambda text: count_hits(text, CODED_VOCAB))
df['coordination_pressure_index'] = (
    0.34 * df['repeat_contact_count']
    + 0.26 * df['tower_handoff_count']
    + 0.20 * df['prior_case_mentions']
    + 0.20 * df['urgent_term_count']
).round(3)
df['covert_language_index'] = (
    (df['coded_term_count'] / df['transcript_token_count'].clip(lower=1))
    + 1.7 * df['silence_ratio']
    + 0.35 * (df['line_type'] == 'burner_phone').astype(float)
    + 0.25 * (df['language_pattern'] == 'code_switched').astype(float)
).round(3)
df['night_activity_flag'] = ((df['call_hour'] >= 22) | (df['call_hour'] <= 4)).astype(int)

feature_table = pd.DataFrame(
    [
        {
            'engineered_feature': 'transcript_token_count',
            'formula': 'len(tokenize(transcript))',
            'forensic_rationale': 'Distinguishes short burst calls from longer coordination narratives.',
        },
        {
            'engineered_feature': 'urgent_term_count',
            'formula': 'count(tokens in urgency vocabulary)',
            'forensic_rationale': 'Captures timing pressure and active movement language.',
        },
        {
            'engineered_feature': 'coded_term_count',
            'formula': 'count(tokens in coded vocabulary)',
            'forensic_rationale': 'Flags euphemistic or obfuscated object references.',
        },
        {
            'engineered_feature': 'coordination_pressure_index',
            'formula': '0.34*repeat_contact_count + 0.26*tower_handoff_count + 0.20*prior_case_mentions + 0.20*urgent_term_count',
            'forensic_rationale': 'Blends frequency, movement, history, and urgency into a triage signal.',
        },
        {
            'engineered_feature': 'covert_language_index',
            'formula': '(coded_term_count/transcript_token_count) + 1.7*silence_ratio + burner/code-switch boosts',
            'forensic_rationale': 'Quantifies coded vocabulary intensity under suspicious call conditions.',
        },
        {
            'engineered_feature': 'night_activity_flag',
            'formula': '1 if call_hour >= 22 or <= 4 else 0',
            'forensic_rationale': 'Marks off-hours contact patterns common in covert coordination.',
        },
    ]
)
print('Feature engineering table:')
print(feature_table.to_string(index=False))

plt.figure(figsize=(12, 4))
sns.boxplot(data=df, x=TARGET, y='covert_language_index', order=CLASS_LABELS, color='#88CCEE')
plt.title('Covert language index by intent')
plt.xticks(rotation=20)
plt.tight_layout()


## Step 5 - Baseline Benchmark

Start with a transparent text-plus-metadata baseline so the PyTorch upgrade has a meaningful comparison point.


In [ ]:
X = df[['case_id'] + MODEL_FEATURES].copy()
y = df[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train,
)

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=280, ngram_range=(1, 2), min_df=2), 'transcript'),
        ('num', StandardScaler(), RAW_NUMERIC_FEATURES + ENGINEERED_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL_FEATURES),
    ]
)

baseline_model = Pipeline(
    steps=[
        ('prep', baseline_preprocessor),
        ('model', LogisticRegression(max_iter=3000, class_weight='balanced')),
    ]
)
baseline_model.fit(X_train[MODEL_FEATURES], y_train)
baseline_pred = baseline_model.predict(X_test[MODEL_FEATURES])
baseline_prob = baseline_model.predict_proba(X_test[MODEL_FEATURES])
baseline_f1 = f1_score(y_test, baseline_pred, average='macro')
baseline_balanced_acc = balanced_accuracy_score(y_test, baseline_pred)

plt.figure(figsize=(12, 4))
baseline_confidence = baseline_prob.max(axis=1)
sns.histplot(baseline_confidence, bins=20, kde=True, color='#4C72B0')
plt.title('Baseline model confidence')
plt.xlabel('Predicted probability of selected intent')
plt.tight_layout()

print(f'Baseline macro F1: {baseline_f1:.3f}')
print(f'Baseline balanced accuracy: {baseline_balanced_acc:.3f}')


## Step 6 - Advanced Model

Train a stronger neural classifier after the baseline establishes the minimum useful performance level. Use PyTorch when available and fall back to a scikit-learn MLP when it is not, so the notebook remains runnable.


In [ ]:
fitted_preprocessor = baseline_model.named_steps['prep']


def to_dense(matrix):
    return matrix.toarray() if hasattr(matrix, 'toarray') else np.asarray(matrix)


X_train_vec = to_dense(fitted_preprocessor.transform(X_train[MODEL_FEATURES]))
X_valid_vec = to_dense(fitted_preprocessor.transform(X_valid[MODEL_FEATURES]))
X_test_vec = to_dense(fitted_preprocessor.transform(X_test[MODEL_FEATURES]))

class_names = list(baseline_model.classes_)
label_to_index = {label: idx for idx, label in enumerate(class_names)}
y_train_idx = np.array([label_to_index[label] for label in y_train])
y_valid_idx = np.array([label_to_index[label] for label in y_valid])
history = []

if TORCH_AVAILABLE:
    train_loader = DataLoader(
        TensorDataset(torch.tensor(X_train_vec, dtype=torch.float32), torch.tensor(y_train_idx, dtype=torch.long)),
        batch_size=128,
        shuffle=True,
    )
    valid_tensor = torch.tensor(X_valid_vec, dtype=torch.float32, device=DEVICE)

    class IntentMLP(nn.Module):
        def __init__(self, input_dim: int, output_dim: int):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.25),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.15),
                nn.Linear(64, output_dim),
            )

        def forward(self, features: torch.Tensor) -> torch.Tensor:
            return self.net(features)

    advanced_model = IntentMLP(X_train_vec.shape[1], len(class_names)).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(advanced_model.parameters(), lr=0.002, weight_decay=1e-4)

    best_state = None
    best_valid_f1 = -1.0
    patience = 5
    wait = 0

    for epoch in range(1, 36):
        advanced_model.train()
        running_loss = 0.0
        sample_count = 0
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
            optimizer.zero_grad()
            logits = advanced_model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
            sample_count += batch_x.size(0)

        advanced_model.eval()
        with torch.no_grad():
            valid_logits = advanced_model(valid_tensor)
            valid_pred_idx = valid_logits.argmax(dim=1).cpu().numpy()
        valid_pred = [class_names[idx] for idx in valid_pred_idx]
        valid_f1 = f1_score(y_valid, valid_pred, average='macro')
        epoch_loss = running_loss / max(sample_count, 1)
        history.append({'epoch': epoch, 'train_loss': epoch_loss, 'valid_macro_f1': valid_f1})

        if valid_f1 > best_valid_f1 + 1e-4:
            best_valid_f1 = valid_f1
            best_state = copy.deepcopy(advanced_model.state_dict())
            wait = 0
        else:
            wait += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f'Epoch {epoch:02d} | train_loss={epoch_loss:.4f} | valid_macro_f1={valid_f1:.3f}')
        if wait >= patience:
            print(f'Early stopping at epoch {epoch}')
            break

    advanced_model.load_state_dict(best_state)
    advanced_model.eval()

    def advanced_predict_proba(frame: pd.DataFrame) -> np.ndarray:
        matrix = to_dense(fitted_preprocessor.transform(frame[MODEL_FEATURES]))
        tensor = torch.tensor(matrix, dtype=torch.float32, device=DEVICE)
        with torch.no_grad():
            return torch.softmax(advanced_model(tensor), dim=1).cpu().numpy()

else:
    advanced_model = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation='relu',
        alpha=1e-4,
        batch_size=128,
        learning_rate_init=0.002,
        max_iter=250,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42,
    )
    advanced_model.fit(X_train_vec, y_train_idx)
    validation_scores = list(getattr(advanced_model, 'validation_scores_', []))
    history = [
        {
            'epoch': idx + 1,
            'train_loss': float(loss),
            'valid_macro_f1': float(validation_scores[idx]) if idx < len(validation_scores) else np.nan,
        }
        for idx, loss in enumerate(advanced_model.loss_curve_)
    ]

    def advanced_predict_proba(frame: pd.DataFrame) -> np.ndarray:
        matrix = to_dense(fitted_preprocessor.transform(frame[MODEL_FEATURES]))
        probs = advanced_model.predict_proba(matrix)
        if list(advanced_model.classes_) != list(range(len(class_names))):
            order = [list(advanced_model.classes_).index(idx) for idx in range(len(class_names))]
            probs = probs[:, order]
        return probs

advanced_prob = advanced_predict_proba(X_test)
advanced_pred = [class_names[idx] for idx in advanced_prob.argmax(axis=1)]
advanced_f1 = f1_score(y_test, advanced_pred, average='macro')
advanced_balanced_acc = balanced_accuracy_score(y_test, advanced_pred)

print(f'Advanced backend: {ADVANCED_BACKEND}')
print(f'Advanced macro F1: {advanced_f1:.3f}')
print(f'Advanced balanced accuracy: {advanced_balanced_acc:.3f}')


## Step 7 - Evaluation and Threshold Design

Compare the baseline and PyTorch model, then inspect where confidence is thin enough to require human review.


In [ ]:
print('Classification report:')
print(classification_report(y_test, advanced_pred, zero_division=0))

history_df = pd.DataFrame(history)
test_confidence = advanced_prob.max(axis=1)
uncertain_cases = X_test[['case_id', 'transcript']].copy()
uncertain_cases['actual_intent'] = y_test.values
uncertain_cases['predicted_intent'] = advanced_pred
uncertain_cases['confidence'] = test_confidence.round(3)
uncertain_cases = uncertain_cases.sort_values('confidence').head(8)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
cm = confusion_matrix(y_test, advanced_pred, labels=class_names)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f'{ADVANCED_BACKEND} confusion matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

comparison = pd.DataFrame(
    {
        'model': ['baseline', 'advanced'],
        'macro_f1': [baseline_f1, advanced_f1],
        'balanced_accuracy': [baseline_balanced_acc, advanced_balanced_acc],
    }
)
comparison.set_index('model').plot(kind='bar', color=['#55A868', '#C44E52'], ax=axes[1])
axes[1].set_ylim(0, 1)
axes[1].set_title('Model comparison')
axes[1].set_ylabel('Score')
plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.lineplot(data=history_df, x='epoch', y='train_loss', marker='o', ax=axes[0], color='#4C72B0')
axes[0].set_title('Training loss')

sns.lineplot(data=history_df, x='epoch', y='valid_macro_f1', marker='o', ax=axes[1], color='#8172B3')
axes[1].set_title('Validation score proxy')
axes[1].set_ylim(0, 1)
plt.tight_layout()

plt.figure(figsize=(12, 4))
sns.histplot(test_confidence, bins=18, kde=True, color='#DD8452')
plt.title(f'{ADVANCED_BACKEND} test confidence profile')
plt.xlabel('Max class probability')
plt.tight_layout()

print('Lowest-confidence review queue:')
print(uncertain_cases.to_string(index=False))


## Step 8 - Explainability with Feature Importance and Partial Dependence

Expose which transcript cues push the transparent baseline and how the stronger model reacts to covert-language intensity.


In [ ]:
feature_names = baseline_model.named_steps['prep'].get_feature_names_out()
coefficient_matrix = baseline_model.named_steps['model'].coef_

token_rows = []
for class_idx, class_name in enumerate(class_names):
    class_df = pd.DataFrame(
        {
            'intent': class_name,
            'feature': feature_names,
            'coefficient': coefficient_matrix[class_idx],
        }
    )
    class_df = class_df[class_df['feature'].str.startswith('text__')].copy()
    class_df['feature'] = class_df['feature'].str.replace('text__', '', regex=False)
    token_rows.append(class_df.nlargest(6, 'coefficient'))
top_token_df = pd.concat(token_rows, ignore_index=True)
print('Top baseline transcript features by class:')
print(top_token_df.to_string(index=False))

focus_top_tokens = top_token_df[top_token_df['intent'] == FOCUS_LABEL].sort_values('coefficient', ascending=True)
focus_index = class_names.index(FOCUS_LABEL)
pd_grid = np.linspace(df['covert_language_index'].quantile(0.05), df['covert_language_index'].quantile(0.95), 14)
pd_values = []
for value in pd_grid:
    temp_frame = X_test.copy()
    temp_frame['covert_language_index'] = value
    pd_values.append(advanced_predict_proba(temp_frame)[:, focus_index].mean())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=focus_top_tokens, x='coefficient', y='feature', color='#4C72B0', ax=axes[0])
axes[0].set_title('Top logistic features for coded_exchange')

axes[1].plot(pd_grid, pd_values, color='#DD8452', linewidth=2.5)
axes[1].set_title('PyTorch sensitivity to covert_language_index')
axes[1].set_xlabel('covert_language_index')
axes[1].set_ylabel('Average P(coded_exchange)')
plt.tight_layout()


## Step 9 - Operational Triage Function

Package the notebook into a reusable analyst-facing function that scores a new call, returns the likely intent, and recommends the next review action.


In [ ]:
numeric_fill = {feature: float(df[feature].median()) for feature in RAW_NUMERIC_FEATURES}
categorical_fill = {feature: df[feature].mode().iloc[0] for feature in CATEGORICAL_FEATURES}
default_transcript = 'routine follow-up call with no unusual coded references'

def enrich_features(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy()
    enriched['transcript'] = enriched['transcript'].fillna(default_transcript)
    for feature, value in numeric_fill.items():
        if feature not in enriched:
            enriched[feature] = value
        enriched[feature] = enriched[feature].fillna(value)
    for feature, value in categorical_fill.items():
        if feature not in enriched:
            enriched[feature] = value
        enriched[feature] = enriched[feature].fillna(value)

    enriched['transcript_token_count'] = enriched['transcript'].apply(lambda text: len(tokenize(text)))
    enriched['urgent_term_count'] = enriched['transcript'].apply(lambda text: count_hits(text, URGENCY_VOCAB))
    enriched['coded_term_count'] = enriched['transcript'].apply(lambda text: count_hits(text, CODED_VOCAB))
    enriched['coordination_pressure_index'] = (
        0.34 * enriched['repeat_contact_count']
        + 0.26 * enriched['tower_handoff_count']
        + 0.20 * enriched['prior_case_mentions']
        + 0.20 * enriched['urgent_term_count']
    ).round(3)
    enriched['covert_language_index'] = (
        (enriched['coded_term_count'] / enriched['transcript_token_count'].clip(lower=1))
        + 1.7 * enriched['silence_ratio']
        + 0.35 * (enriched['line_type'] == 'burner_phone').astype(float)
        + 0.25 * (enriched['language_pattern'] == 'code_switched').astype(float)
    ).round(3)
    enriched['night_activity_flag'] = ((enriched['call_hour'] >= 22) | (enriched['call_hour'] <= 4)).astype(int)
    return enriched

ACTION_MAP = {
    'planning': 'Correlate the handset with call-detail records and keep the intercept window open for follow-up coordination.',
    'execution_signal': 'Escalate immediately, preserve associated device images, and prioritize fast-turn metadata expansion.',
    'post_event': 'Preserve deletion-prone artifacts, compare with prior device snapshots, and examine cleanup behavior.',
    'coded_exchange': 'Route for analyst review, glossary expansion, and contact-graph pivoting around coded terms.',
}

def triage_police_call_transcript(evidence: dict, verbose: bool = True):
    record = pd.DataFrame([evidence])
    if 'case_id' not in record:
        record['case_id'] = 'LIVE-REVIEW'
    record = enrich_features(record)
    baseline_label = baseline_model.predict(record[MODEL_FEATURES])[0]
    probabilities = advanced_predict_proba(record)[0]
    predicted_idx = int(np.argmax(probabilities))
    predicted_label = class_names[predicted_idx]
    confidence = float(np.max(probabilities))
    result = {
        'predicted_intent': predicted_label,
        'confidence': round(confidence, 3),
        'baseline_label': baseline_label,
        'recommended_action': ACTION_MAP[predicted_label],
        'human_review_required': confidence < 0.75,
    }
    if verbose:
        print(pd.Series(result).to_string())
    return result

example_case = {
    'case_id': 'LIVE-2047',
    'call_hour': 23,
    'call_duration_seconds': 124,
    'repeat_contact_count': 4,
    'tower_handoff_count': 1,
    'prior_case_mentions': 1,
    'silence_ratio': 0.34,
    'speaker_overlap_ratio': 0.12,
    'line_type': 'burner_phone',
    'language_pattern': 'code_switched',
    'contact_relationship': 'hub_contact',
    'transcript': 'the green folders are ready for pickup bring the blue package after rain keep this short and routine',
}
analyst_result = triage_police_call_transcript(example_case, verbose=True)

example_probs = pd.Series(
    advanced_predict_proba(enrich_features(pd.DataFrame([example_case])))[0],
    index=class_names,
).sort_values()
plt.figure(figsize=(10, 4))
example_probs.plot(kind='barh', color='#64B5CD')
plt.title('Example case intent probabilities')
plt.xlabel('Probability')
plt.tight_layout()


## Step 10 - Summary and Investigative Handoff

Capture the final modeling outcome, key caveats, and the operational function an analyst would reuse downstream.


In [ ]:
summary_table = pd.DataFrame(
    [
        {'dimension': 'Notebook', 'value': NOTEBOOK_TITLE},
        {'dimension': 'Topic', 'value': TOPIC_NAME},
        {'dimension': 'Target', 'value': TARGET},
        {'dimension': 'Dataset rows', 'value': len(df)},
        {'dimension': 'Advanced backend', 'value': ADVANCED_BACKEND},
        {'dimension': 'Baseline macro F1', 'value': round(float(baseline_f1), 3)},
        {'dimension': 'Advanced macro F1', 'value': round(float(advanced_f1), 3)},
        {'dimension': 'Advanced balanced accuracy', 'value': round(float(advanced_balanced_acc), 3)},
        {'dimension': 'Explainability focus', 'value': focus_top_tokens.iloc[-1]['feature']},
        {'dimension': 'Operational function', 'value': 'triage_police_call_transcript'},
        {'dimension': 'Primary analyst decision', 'value': 'whether to preserve live intercept windows, subpoena additional carrier metadata, expand device imaging, or prioritize transcript translation'},
    ]
)
print('Summary table:')
print(summary_table.to_string(index=False))
print()
print('Limitations:')
limitations = [
    'Synthetic transcripts simplify slang drift, accent variation, and transcription error.',
    'Coded language changes over time and requires analyst-maintained glossaries.',
    'Predicted intent is decision support only and should be corroborated with carrier, handset, and chain-of-custody evidence.',
]
for item in limitations:
    print(f'- {item}')
